In [0]:
%run ./utils/logger

In [0]:
run_id = get_run_id()
print(run_id)


In [0]:
dbutils.widgets.text('catalog','commerce_stage_dev')
dbutils.widgets.text('schema','silver')
dbutils.widgets.text("env", "dev")

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
env = dbutils.widgets.get("env")

In [0]:
order_items_df = spark.table(f'commerce_raw_{env}.bronze.order_items')

In [0]:
display(order_items_df)

In [0]:
from pyspark.sql.functions import col
order_items_filter_df = order_items_df.filter(col("order_id").isNotNull())

In [0]:
order_returns_clean_df = order_items_filter_df.distinct()

In [0]:
order_returns_clean_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{schema}.order_items_temp1")

In [0]:
%sql
select * from order_items_temp1

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.order_items_stage_dedup AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id, product_id
               ORDER BY ingestion_time DESC
           ) AS rn
    FROM order_items_temp1
)
WHERE rn = 1
""")


In [0]:
df = spark.sql(f'describe table extended commerce_stage_{env}.{schema}.order_items_stage_dedup')
display(df)

In [0]:
spark.sql(f"""
MERGE INTO {catalog}.{schema}.order_detail_stage tgt
USING {catalog}.{schema}.order_items_stage_dedup src

ON tgt.order_id = src.order_id
AND tgt.product_id = src.product_id

WHEN MATCHED THEN
UPDATE SET
    tgt.quantity = src.quantity,
    tgt.unit_price = src.unit_price,
    tgt.updated_ts = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    order_id,
    product_id,
    quantity,
    unit_price,
    created_ts,
    updated_ts
)
VALUES (
    src.order_id,
    src.product_id,
    src.quantity,
    src.unit_price,
    current_timestamp(),
    current_timestamp()
)
""")

In [0]:
spark.sql(f"""
          drop table if exists commerce_stage_{env}.{schema}.order_items_temp1
          """)

In [0]:
spark.sql(f"""
          drop table if exists commerce_stage_{env}.{schema}.order_items_stage_dedup
          """)